# The SOHO Mission: An Overview — Implementation / 구현

**Paper**: Domingo, V., Fleck, B., and Poland, A.I. (1995). "The SOHO Mission: An Overview." *Solar Physics*, 162, 1–37. [DOI: 10.1007/BF00733425]

이 노트북은 SOHO 미션 개요 논문의 핵심 개념들을 구현합니다. L1 라그랑주 점 계산, 헤일로 궤도 시뮬레이션, 기기 배치 시각화, 코로나그래프 커버리지, 텔레메트리 예산, 태양 주기 커버리지를 다룹니다.

This notebook implements key concepts from the SOHO mission overview paper: L1 Lagrange point calculation, halo orbit simulation, instrument suite visualization, coronagraph coverage, telemetry budget, and solar cycle coverage.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import brentq
from mpl_toolkits.mplot3d import Axes3D

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['figure.dpi'] = 100

## Part 1: L1 Lagrange Point Calculation / L1 라그랑주 점 계산

SOHO는 태양-지구 시스템의 L1 라그랑주 점 근처에서 운용됩니다. L1은 태양과 지구 사이에 위치하며, 원형 제한 삼체 문제(CR3BP)에서 유도됩니다.

SOHO operates near the L1 Lagrange point of the Sun-Earth system. L1 is located between the Sun and Earth, derived from the Circular Restricted Three-Body Problem (CR3BP).

**정확한 방정식 / Exact equation:**
$$x^3 - (3-\mu)x^2 + (3-2\mu)x - \mu = 0$$

여기서 $\mu = M_\oplus / (M_\odot + M_\oplus) \approx 3.003 \times 10^{-6}$

**근사 해 / Approximate solution:**
$$r_{L1} \approx R \left(\frac{\mu}{3}\right)^{1/3} \approx 1.5 \times 10^6 \text{ km}$$

In [ ]:
# --- Physical Constants ---
M_sun = 1.989e30       # Solar mass [kg]
M_earth = 5.972e24     # Earth mass [kg]
R_se = 1.496e8         # Sun-Earth distance [km] (1 AU)
G = 6.674e-11          # Gravitational constant [m^3 kg^-1 s^-2]

# Mass ratio for CR3BP.
mu = M_earth / (M_sun + M_earth)
print(f"Mass ratio mu = {mu:.6e}")


def collinear_equilibrium(x, mu):
    """Equation for collinear Lagrange points: dOmega/dx = 0 on y=0.

    In the rotating frame (normalized units), Sun at -mu, Earth at 1-mu.
    The equilibrium condition on the x-axis is:
    x - (1-mu)*(x+mu)/|x+mu|^3 - mu*(x-1+mu)/|x-1+mu|^3 = 0

    Args:
        x: Position along Sun-Earth line (normalized).
        mu: Mass ratio M_earth / (M_sun + M_earth).

    Returns:
        Residual of the equilibrium equation.
    """
    r1 = abs(x + mu)
    r2 = abs(x - 1 + mu)
    return x - (1 - mu) * (x + mu) / r1**3 - mu * (x - 1 + mu) / r2**3


# --- Exact solution: find L1 using Brent's method ---
# L1 is between Sun and Earth, close to Earth (x ~ 0.99 in normalized units).
x_l1_exact = brentq(collinear_equilibrium, 0.985, 0.995, args=(mu,))
r_l1_exact_km = (1 - mu - x_l1_exact) * R_se  # Distance from Earth

# --- Approximate solution ---
r_l1_approx_km = R_se * (mu / 3) ** (1.0 / 3.0)

print(f"\n--- L1 Location Results ---")
print(f"Exact (root-finding):  x_L1 = {x_l1_exact:.10f} (normalized)")
print(f"  Distance from Earth: {r_l1_exact_km:.0f} km = {r_l1_exact_km/1e6:.3f} million km")
print(f"  Distance from Earth: {r_l1_exact_km/R_se:.6f} AU")
print(f"\nApproximate (mu/3)^(1/3): {r_l1_approx_km:.0f} km = {r_l1_approx_km/1e6:.3f} million km")
print(f"\nDifference: {abs(r_l1_exact_km - r_l1_approx_km):.0f} km "
      f"({abs(r_l1_exact_km - r_l1_approx_km)/r_l1_exact_km*100:.2f}%)")
print(f"\nPaper value: ~1.5 million km")

In [ ]:
# --- Effective Potential in the Rotating Frame ---
# Omega(x, y) = -(1/2)(x^2 + y^2) - (1-mu)/r1 - mu/r2
# where r1 = distance to Sun, r2 = distance to Earth (normalized units).

def effective_potential(x, y, mu):
    """Compute the effective potential in the CR3BP rotating frame.

    Args:
        x: x-coordinate (normalized, Sun-Earth line).
        y: y-coordinate (normalized, perpendicular).
        mu: Mass ratio M_earth / (M_sun + M_earth).

    Returns:
        Effective potential Omega(x, y).
    """
    r1 = np.sqrt((x + mu) ** 2 + y ** 2)        # Distance to Sun (at -mu)
    r2 = np.sqrt((x - 1 + mu) ** 2 + y ** 2)    # Distance to Earth (at 1-mu)
    # Avoid division by zero.
    r1 = np.maximum(r1, 1e-10)
    r2 = np.maximum(r2, 1e-10)
    return -0.5 * (x ** 2 + y ** 2) - (1 - mu) / r1 - mu / r2


# Find all 5 Lagrange points using the correct equilibrium equation.
# L1: between Sun and Earth.
x_L1 = brentq(collinear_equilibrium, 0.985, 0.995, args=(mu,))
# L2: beyond Earth (away from Sun).
x_L2 = brentq(collinear_equilibrium, 1.001, 1.02, args=(mu,))
# L3: beyond Sun (opposite side from Earth).
x_L3 = brentq(collinear_equilibrium, -1.1, -0.99, args=(mu,))
# L4 and L5: equilateral triangle points.
x_L4, y_L4 = 0.5 - mu, np.sqrt(3) / 2
x_L5, y_L5 = 0.5 - mu, -np.sqrt(3) / 2

lagrange_points = {
    'L1': (x_L1, 0),
    'L2': (x_L2, 0),
    'L3': (x_L3, 0),
    'L4': (x_L4, y_L4),
    'L5': (x_L5, y_L5),
}

for name, (xp, yp) in lagrange_points.items():
    print(f"{name}: x = {xp:.6f}, y = {yp:.6f}")

# Plot effective potential contours.
N = 500
x_grid = np.linspace(-1.8, 1.8, N)
y_grid = np.linspace(-1.8, 1.8, N)
X, Y = np.meshgrid(x_grid, y_grid)
Z = effective_potential(X, Y, mu)

# Clip for visualization (potential diverges near bodies).
Z_clip = np.clip(Z, -4, -1.4)

fig, ax = plt.subplots(figsize=(10, 9))
levels = np.linspace(-3.5, -1.45, 50)
cs = ax.contourf(X, Y, Z_clip, levels=levels, cmap='RdYlBu_r')
ax.contour(X, Y, Z_clip, levels=levels, colors='k', linewidths=0.2, alpha=0.3)
plt.colorbar(cs, ax=ax, label='Effective Potential (normalized)')

# Mark Lagrange points.
colors_lp = {'L1': 'red', 'L2': 'blue', 'L3': 'green', 'L4': 'purple', 'L5': 'purple'}
for name, (xp, yp) in lagrange_points.items():
    ax.plot(xp, yp, 'o', color=colors_lp[name], markersize=8, zorder=5)
    offset = (10, 10) if name != 'L3' else (-20, 10)
    ax.annotate(name, (xp, yp), textcoords='offset points',
                xytext=offset, fontsize=12, fontweight='bold', color=colors_lp[name])

# Mark Sun and Earth.
ax.plot(-mu, 0, '*', color='yellow', markersize=20, markeredgecolor='orange',
        markeredgewidth=1, zorder=6, label='Sun')
ax.plot(1 - mu, 0, 'o', color='cyan', markersize=10, markeredgecolor='blue',
        markeredgewidth=1, zorder=6, label='Earth')

ax.set_xlabel('x (normalized, Sun-Earth line)')
ax.set_ylabel('y (normalized)')
ax.set_title('Effective Potential in the Rotating Frame (CR3BP)\nwith L1-L5 Lagrange Points')
ax.legend(loc='upper right', fontsize=11)
ax.set_aspect('equal')
ax.set_xlim(-1.8, 1.8)
ax.set_ylim(-1.8, 1.8)
plt.tight_layout()
plt.show()

print(f"\nSOHO orbits near L1 at x = {x_L1:.6f} (normalized)")
print(f"  = {(1 - mu - x_L1) * R_se / 1e6:.2f} million km sunward of Earth")

## Part 2: Halo Orbit Simulation / 헤일로 궤도 시뮬레이션

SOHO는 L1 점 주위의 헤일로 궤도를 따라 운동합니다. 이 궤도는 SOHO가 지구에서 볼 때 태양 앞을 지나가지 않도록 설계되었습니다 (통신 간섭 방지).

SOHO follows a halo orbit around L1. This orbit is designed so that SOHO never passes directly in front of the Sun as seen from Earth (avoiding communication interference).

**논문 값 / Paper values** (Section 3.2, Fig. 13):
- 반지름 / Semi-diameters: ~200,000 km (x, Sun-Earth), ~650,000 km (y, ecliptic), ~200,000 km (z, out of ecliptic)
- 주기 / Period: 180 days
- 속도 변화 / Velocity variation: ~$\pm$16 m/s/day

**Lissajous 궤도 근사 / Lissajous orbit approximation:**
$$x(t) = A_x \cos(\omega t + \phi_x)$$
$$y(t) = A_y \cos(\omega t + \phi_y)$$
$$z(t) = A_z \cos(\omega_z t + \phi_z)$$

In [ ]:
# --- SOHO Halo Orbit Parameters ---
# Semi-diameters from the paper (half-amplitudes).
Ax = 200_000   # km, along Sun-Earth line
Ay = 650_000   # km, in ecliptic plane perpendicular to Sun-Earth line
Az = 200_000   # km, out of ecliptic plane

T_halo = 180   # Orbital period [days]
omega = 2 * np.pi / T_halo  # Angular frequency [rad/day]

# For a true halo orbit, the in-plane and out-of-plane frequencies are coupled.
# For Lissajous, we allow slightly different z-frequency.
# Here we use the same frequency for a halo orbit (phase-locked).
omega_z = omega  # Same frequency for halo orbit

# Phase offsets: halo orbit has a specific phase relationship.
phi_x = 0
phi_y = np.pi / 2  # 90 degree phase offset gives elliptical projection
phi_z = 0           # In-phase with x for halo orbit

# Time array: 2 full orbits.
t = np.linspace(0, 2 * T_halo, 2000)

# Parametric orbit.
x_orbit = Ax * np.cos(omega * t + phi_x)
y_orbit = Ay * np.cos(omega * t + phi_y)
z_orbit = Az * np.cos(omega_z * t + phi_z)

# --- 3D Orbit Plot ---
fig = plt.figure(figsize=(12, 10))
ax3d = fig.add_subplot(111, projection='3d')
ax3d.plot(x_orbit / 1e3, y_orbit / 1e3, z_orbit / 1e3,
          'b-', linewidth=1, alpha=0.8, label='SOHO Halo Orbit')
ax3d.scatter([0], [0], [0], color='red', s=100, marker='*', label='L1 Point')

# Mark starting position.
ax3d.scatter([x_orbit[0] / 1e3], [y_orbit[0] / 1e3], [z_orbit[0] / 1e3],
             color='green', s=80, marker='o', label='Start', zorder=5)

ax3d.set_xlabel('X (Sun-Earth) [x1000 km]')
ax3d.set_ylabel('Y (Ecliptic plane) [x1000 km]')
ax3d.set_zlabel('Z (Out of ecliptic) [x1000 km]')
ax3d.set_title('SOHO Halo Orbit around L1 (2 periods = 360 days)')
ax3d.legend(loc='upper left')
plt.tight_layout()
plt.show()

# --- Projection Plots ---
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# X-Y projection (view from above ecliptic).
axes[0].plot(x_orbit / 1e3, y_orbit / 1e3, 'b-', linewidth=1)
axes[0].plot(0, 0, 'r*', markersize=15, label='L1')
axes[0].set_xlabel('X (Sun-Earth) [x1000 km]')
axes[0].set_ylabel('Y (Ecliptic) [x1000 km]')
axes[0].set_title('X-Y Projection (top view)')
axes[0].set_aspect('equal')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# X-Z projection (view from ecliptic pole).
axes[1].plot(x_orbit / 1e3, z_orbit / 1e3, 'b-', linewidth=1)
axes[1].plot(0, 0, 'r*', markersize=15, label='L1')
axes[1].set_xlabel('X (Sun-Earth) [x1000 km]')
axes[1].set_ylabel('Z (Out of ecliptic) [x1000 km]')
axes[1].set_title('X-Z Projection (side view)')
axes[1].set_aspect('equal')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Y-Z projection (view from Sun/Earth, "sky plane").
axes[2].plot(y_orbit / 1e3, z_orbit / 1e3, 'b-', linewidth=1)
axes[2].plot(0, 0, 'r*', markersize=15, label='L1')
axes[2].set_xlabel('Y (Ecliptic) [x1000 km]')
axes[2].set_ylabel('Z (Out of ecliptic) [x1000 km]')
axes[2].set_title('Y-Z Projection (Earth view)')
axes[2].set_aspect('equal')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# --- Velocity Variation (Fig. 13 concept) ---
# Compute velocity components from the parametric orbit derivatives.
# dx/dt, dy/dt, dz/dt in km/day, then convert to m/s.
vx = -Ax * omega * np.sin(omega * t + phi_x)  # km/day
vy = -Ay * omega * np.sin(omega * t + phi_y)  # km/day
vz = -Az * omega_z * np.sin(omega_z * t + phi_z)  # km/day

# Total velocity in km/day, convert to m/s (1 km/day = 1000/86400 m/s).
km_day_to_m_s = 1000.0 / 86400.0
v_total = np.sqrt(vx**2 + vy**2 + vz**2) * km_day_to_m_s

# Velocity change (acceleration) in m/s/day via finite differences.
dt_days = t[1] - t[0]
dv_dt = np.gradient(v_total, dt_days)  # m/s per day

fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

# Velocity magnitude.
axes[0].plot(t, v_total, 'b-', linewidth=1.5)
axes[0].set_ylabel('Orbital Velocity [m/s]')
axes[0].set_title('SOHO Halo Orbit Velocity (cf. Fig. 13 concept)')
axes[0].grid(True, alpha=0.3)
axes[0].axhline(y=np.mean(v_total), color='r', linestyle='--', alpha=0.5,
                label=f'Mean: {np.mean(v_total):.1f} m/s')
axes[0].legend()

# Velocity change rate (acceleration).
axes[1].plot(t, dv_dt, 'r-', linewidth=1.5)
axes[1].set_xlabel('Time [days]')
axes[1].set_ylabel('dV/dt [m/s/day]')
axes[1].set_title('Velocity Change Rate')
axes[1].grid(True, alpha=0.3)
axes[1].axhline(y=16, color='gray', linestyle=':', alpha=0.7, label='Paper: +/-16 m/s/day')
axes[1].axhline(y=-16, color='gray', linestyle=':', alpha=0.7)
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"Velocity range: {v_total.min():.1f} - {v_total.max():.1f} m/s")
print(f"dV/dt range: {dv_dt.min():.1f} to {dv_dt.max():.1f} m/s/day")
print(f"Paper reference: ~+/-16 m/s/day smooth variation")

## Part 3: SOHO Instrument Suite Visualization / SOHO 기기 배치 시각화

SOHO는 3개 과학 분야의 12개 기기를 탑재하고 있습니다:

SOHO carries 12 instruments across three scientific domains:

**태양 진동 / Helioseismology**: GOLF, VIRGO, MDI
**태양 대기 / Solar Atmosphere**: SUMER, CDS, EIT, UVCS, LASCO, SWAN
**태양풍 / Solar Wind**: CELIAS, COSTEP, ERNE

아래에서 (a) 파장/시야각 커버리지, (b) 데이터 전송률 분배, (c) 태양 진동 기기의 공간 분해능을 시각화합니다.

Below we visualize (a) wavelength/FOV coverage, (b) data rate distribution, and (c) helioseismology instrument spatial resolution.

In [ ]:
# --- (a) Wavelength Coverage Chart ---
# Approximate wavelength ranges for each instrument (in Angstroms).
# This captures the spirit of Fig. 6 from the paper.

instruments_wave = {
    'GOLF':    {'range': (7699, 7699), 'type': 'Helioseismology',
                'desc': 'Na D line (global oscillations)'},
    'VIRGO':   {'range': (4000, 8600), 'type': 'Helioseismology',
                'desc': 'Broadband photometry'},
    'MDI':     {'range': (6768, 6768), 'type': 'Helioseismology',
                'desc': 'Ni I line (Dopplergrams)'},
    'SUMER':   {'range': (500, 1610), 'type': 'Atmosphere',
                'desc': 'UV spectrometer'},
    'CDS':     {'range': (150, 800), 'type': 'Atmosphere',
                'desc': 'EUV spectrometer'},
    'EIT':     {'range': (171, 304), 'type': 'Atmosphere',
                'desc': 'EUV imager (4 bands)'},
    'UVCS':    {'range': (500, 1270), 'type': 'Atmosphere',
                'desc': 'UV coronagraph spectrometer'},
    'LASCO':   {'range': (4000, 7000), 'type': 'Atmosphere',
                'desc': 'White-light coronagraphs'},
    'SWAN':    {'range': (1216, 1216), 'type': 'Atmosphere',
                'desc': 'Ly-alpha (full sky)'},
    'CELIAS':  {'range': (None, None), 'type': 'Solar Wind',
                'desc': 'In-situ particles'},
    'COSTEP':  {'range': (None, None), 'type': 'Solar Wind',
                'desc': 'In-situ particles/electrons'},
    'ERNE':    {'range': (None, None), 'type': 'Solar Wind',
                'desc': 'In-situ energetic particles'},
}

# Color map by instrument type.
type_colors = {
    'Helioseismology': '#2196F3',
    'Atmosphere': '#FF9800',
    'Solar Wind': '#4CAF50',
}

fig, ax = plt.subplots(figsize=(14, 8))

# Plot remote-sensing instruments only (those with wavelength ranges).
remote_instruments = {k: v for k, v in instruments_wave.items()
                      if v['range'][0] is not None}
y_pos = 0
y_labels = []
y_positions = []

for name, info in remote_instruments.items():
    color = type_colors[info['type']]
    wl_min, wl_max = info['range']
    if wl_min == wl_max:
        # Single wavelength: plot as a marker.
        ax.plot(wl_min, y_pos, 'D', color=color, markersize=10, zorder=5)
        ax.annotate(f"{wl_min} A", (wl_min, y_pos),
                    textcoords='offset points', xytext=(10, 0), fontsize=9)
    else:
        # Wavelength range: plot as horizontal bar.
        ax.barh(y_pos, wl_max - wl_min, left=wl_min, height=0.6,
                color=color, alpha=0.7, edgecolor='black', linewidth=0.5)
    y_labels.append(f"{name}\n({info['desc']})")
    y_positions.append(y_pos)
    y_pos += 1

# Add in-situ instruments as text annotation.
ax.text(5000, y_pos + 0.5,
        'In-situ instruments (CELIAS, COSTEP, ERNE):\nParticle detectors, not wavelength-based',
        fontsize=10, style='italic', color='#4CAF50',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='#E8F5E9', alpha=0.8))

ax.set_yticks(y_positions)
ax.set_yticklabels(y_labels, fontsize=9)
ax.set_xlabel('Wavelength [Angstroms]')
ax.set_title('SOHO Instrument Wavelength Coverage (cf. Fig. 6)')
ax.set_xscale('log')
ax.set_xlim(100, 10000)
ax.grid(True, axis='x', alpha=0.3)

# Add legend for instrument types.
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=c, label=t) for t, c in type_colors.items()]
ax.legend(handles=legend_elements, loc='lower right', fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
# --- (b) Data Rate Pie Chart ---
# Telemetry budget from Table II (continuous rate: 40 kbit/s total).
# Using continuous (normal) data rates.

data_rates = {
    'GOLF': 0.16,
    'VIRGO': 0.10,
    'MDI': 5.0,
    'SUMER': 10.5,
    'CDS': 12.0,
    'EIT': 1.0,
    'UVCS': 5.0,
    'LASCO': 4.2,
    'SWAN': 0.2,
    'CELIAS': 1.5,
    'COSTEP': 0.3,
    'ERNE': 0.7,
    'Housekeeping': 1.3,
}

labels = list(data_rates.keys())
sizes = list(data_rates.values())
total = sum(sizes)

# Colors by instrument type.
inst_type_map = {
    'GOLF': 'Helio', 'VIRGO': 'Helio', 'MDI': 'Helio',
    'SUMER': 'Atmo', 'CDS': 'Atmo', 'EIT': 'Atmo',
    'UVCS': 'Atmo', 'LASCO': 'Atmo', 'SWAN': 'Atmo',
    'CELIAS': 'Wind', 'COSTEP': 'Wind', 'ERNE': 'Wind',
    'Housekeeping': 'Other',
}
color_map = {
    'Helio': '#64B5F6', 'Atmo': '#FFB74D',
    'Wind': '#81C784', 'Other': '#E0E0E0',
}
colors = [color_map[inst_type_map[l]] for l in labels]

# Explode the largest slices.
explode = [0.05 if s > 3 else 0 for s in sizes]

fig, ax = plt.subplots(figsize=(10, 10))
wedges, texts, autotexts = ax.pie(
    sizes, labels=labels, autopct=lambda p: f'{p:.1f}%' if p > 2 else '',
    colors=colors, explode=explode, startangle=90,
    textprops={'fontsize': 10}, pctdistance=0.8
)

# Adjust small-slice label positions.
for text in texts:
    text.set_fontsize(9)

ax.set_title(f'SOHO Telemetry Budget (Total: {total:.1f} kbit/s continuous)\n'
             'Data from Table II', fontsize=13)

# Legend by type.
legend_elements = [
    Patch(facecolor='#64B5F6', label='Helioseismology'),
    Patch(facecolor='#FFB74D', label='Solar Atmosphere'),
    Patch(facecolor='#81C784', label='Solar Wind'),
    Patch(facecolor='#E0E0E0', label='Housekeeping'),
]
ax.legend(handles=legend_elements, loc='lower right', fontsize=10)
plt.tight_layout()
plt.show()

print(f"Total continuous telemetry: {total:.2f} kbit/s (budget: 40 kbit/s)")
print(f"\nTop consumers: CDS ({data_rates['CDS']} kbit/s), "
      f"SUMER ({data_rates['SUMER']} kbit/s), MDI ({data_rates['MDI']} kbit/s)")

In [ ]:
# --- (c) Helioseismology Spatial Coverage ---
# Angular degree (l) coverage comparison:
# GOLF: full-disk integrated, sensitive to l=0-3
# VIRGO: 12 pixels (LOI), sensitive to l=0-7
# MDI: 1024x1024 pixels, sensitive to l=0-4500

instruments_helio = {
    'GOLF': {'l_max': 3, 'pixels': '1 (full disk)', 'color': '#1565C0'},
    'VIRGO/LOI': {'l_max': 7, 'pixels': '12', 'color': '#1E88E5'},
    'MDI': {'l_max': 4500, 'pixels': '1024x1024', 'color': '#42A5F5'},
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart: angular degree coverage (log scale).
names = list(instruments_helio.keys())
l_max_vals = [instruments_helio[n]['l_max'] for n in names]
colors_h = [instruments_helio[n]['color'] for n in names]

bars = axes[0].bar(names, l_max_vals, color=colors_h, edgecolor='black', linewidth=0.5)
axes[0].set_yscale('log')
axes[0].set_ylabel('Maximum Angular Degree (l)')
axes[0].set_title('Helioseismology: Angular Degree Coverage')
axes[0].grid(True, axis='y', alpha=0.3)

# Add value labels on bars.
for bar, val in zip(bars, l_max_vals):
    axes[0].text(bar.get_x() + bar.get_width() / 2, val * 1.3,
                 f'l = 0-{val}', ha='center', va='bottom', fontsize=11,
                 fontweight='bold')

# Pixel count comparison.
pixel_counts = [1, 12, 1024**2]
bars2 = axes[1].bar(names, pixel_counts, color=colors_h, edgecolor='black', linewidth=0.5)
axes[1].set_yscale('log')
axes[1].set_ylabel('Number of Spatial Elements')
axes[1].set_title('Helioseismology: Spatial Sampling')
axes[1].grid(True, axis='y', alpha=0.3)

for bar, val, label in zip(bars2, pixel_counts,
                           ['1 pixel', '12 pixels', '1024x1024']):
    axes[1].text(bar.get_x() + bar.get_width() / 2, val * 1.5,
                 label, ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

print("Helioseismology instrument comparison:")
print(f"  GOLF:  l=0-3   (1 pixel, full-disk velocity)")
print(f"  VIRGO: l=0-7   (12 pixels, LOI + SPM)")
print(f"  MDI:   l=0-4500 (1024x1024 pixels, Dopplergrams)")

## Part 4: LASCO Coronagraph Coverage / LASCO 코로나그래프 커버리지

LASCO는 3개의 코로나그래프로 구성되어 있으며, 태양 반경 1.1~30 $R_\odot$ 범위를 커버합니다:

LASCO consists of three coronagraphs covering 1.1-30 $R_\odot$:

- **C1**: 1.1-3 $R_\odot$ (내부 차폐, Fabry-Perot 분광 / internally occulted, Fabry-Perot spectroscopic)
- **C2**: 1.5-6 $R_\odot$ (외부 차폐, 백색광 / externally occulted, white light)
- **C3**: 3-30 $R_\odot$ (외부 차폐, 백색광 / externally occulted, white light)

우주에서의 산란광 수준은 지상보다 극적으로 개선됩니다: 지상 ~$10^{-6}$ $B_\odot$ vs 우주 LASCO C2 ~$10^{-10}$ $B_\odot$.

The stray light level from space is dramatically improved over ground: ground ~$10^{-6}$ $B_\odot$ vs space LASCO C2 ~$10^{-10}$ $B_\odot$.

In [ ]:
# --- LASCO Coronagraph FOV Coverage ---
# Radial coverage of each coronagraph and comparison instruments.

coronagraphs = {
    'Solar Disk':       {'r_min': 0, 'r_max': 1.0, 'color': '#FFF176',
                         'alpha': 0.9},
    'SDO/AIA':          {'r_min': 0, 'r_max': 1.3, 'color': '#CE93D8',
                         'alpha': 0.4},
    'LASCO C1':         {'r_min': 1.1, 'r_max': 3.0, 'color': '#EF5350',
                         'alpha': 0.6},
    'LASCO C2':         {'r_min': 1.5, 'r_max': 6.0, 'color': '#42A5F5',
                         'alpha': 0.6},
    'LASCO C3':         {'r_min': 3.0, 'r_max': 30.0, 'color': '#66BB6A',
                         'alpha': 0.6},
    'K-Cor (ground)':   {'r_min': 1.05, 'r_max': 3.0, 'color': '#FF9800',
                         'alpha': 0.4},
}

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# --- Left: Polar diagram showing nested FOV ---
ax_polar = axes[0]
theta = np.linspace(0, 2 * np.pi, 200)

for name, info in coronagraphs.items():
    r_in = info['r_min']
    r_out = info['r_max']
    # Draw filled annulus.
    theta_fill = np.concatenate([theta, theta[::-1]])
    r_fill = np.concatenate([np.full_like(theta, r_out),
                             np.full_like(theta, r_in)])
    x_fill = r_fill * np.cos(theta_fill)
    y_fill = r_fill * np.sin(theta_fill)
    ax_polar.fill(x_fill, y_fill, color=info['color'], alpha=info['alpha'],
                  label=f"{name} ({r_in}-{r_out} Rs)")

# Draw reference circles.
for r in [1, 3, 6, 10, 20, 30]:
    circle = plt.Circle((0, 0), r, fill=False, color='gray',
                         linestyle=':', linewidth=0.5, alpha=0.5)
    ax_polar.add_patch(circle)
    if r <= 30:
        ax_polar.text(r * 0.707, r * 0.707, f'{r} Rs',
                      fontsize=7, color='gray', alpha=0.7)

ax_polar.set_xlim(-32, 32)
ax_polar.set_ylim(-32, 32)
ax_polar.set_aspect('equal')
ax_polar.set_xlabel('Distance [solar radii]')
ax_polar.set_ylabel('Distance [solar radii]')
ax_polar.set_title('LASCO Coronagraph Coverage (Radial)')
ax_polar.legend(loc='upper right', fontsize=8, framealpha=0.9)

# --- Right: Stray light comparison ---
instruments_stray = {
    'Ground coronagraph\n(typical)': 1e-6,
    'Ground coronagraph\n(best, e.g. K-Cor)': 1e-7,
    'LASCO C1\n(internally occulted)': 1e-8,
    'LASCO C2\n(externally occulted)': 1e-10,
    'LASCO C3\n(externally occulted)': 1e-10,
}

# K-corona brightness at different heights for reference.
# K-corona ~ 10^-6 at 1.5 Rs, 10^-8 at 3 Rs, 10^-10 at 6 Rs.
r_corona = np.array([1.1, 1.5, 2, 3, 5, 6, 10, 20, 30])
# Approximate K-corona brightness (Allen 1973 model).
b_kcorona = 3.67e-5 * r_corona**(-2.5) + 1.93e-6 * r_corona**(-6)

names_s = list(instruments_stray.keys())
values_s = list(instruments_stray.values())
colors_s = ['#FF7043', '#FF9800', '#EF5350', '#42A5F5', '#66BB6A']

ax_stray = axes[1]
bars_s = ax_stray.barh(names_s, values_s, color=colors_s,
                       edgecolor='black', linewidth=0.5, height=0.6)
ax_stray.set_xscale('log')
ax_stray.set_xlabel('Stray Light Level [B_sun]')
ax_stray.set_title('Stray Light: Ground vs Space')
ax_stray.set_xlim(1e-11, 1e-5)
ax_stray.grid(True, axis='x', alpha=0.3)

# Add value labels.
for bar, val in zip(bars_s, values_s):
    ax_stray.text(val * 2, bar.get_y() + bar.get_height() / 2,
                  f'{val:.0e}', va='center', fontsize=9, fontweight='bold')

# Annotate improvement factor.
ax_stray.annotate('10,000x\nimprovement',
                  xy=(1e-10, 3.5), xytext=(1e-8, 4.2),
                  fontsize=11, fontweight='bold', color='red',
                  arrowprops=dict(arrowstyle='->', color='red'),
                  ha='center')

plt.tight_layout()
plt.show()

print("LASCO achieves stray light of ~10^-10 B_sun, a 10,000x improvement")
print("over the best ground-based coronagraphs (~10^-6 B_sun).")

## Part 5: DSN Telemetry and Data Budget / DSN 텔레메트리 및 데이터 예산

SOHO의 데이터 다운링크는 NASA의 Deep Space Network(DSN)을 통해 이루어집니다. 96% 데이터 연속성 목표를 달성하기 위해 온보드 저장과 다중 통신 패스를 사용합니다.

SOHO's data downlink uses NASA's Deep Space Network (DSN). On-board storage and multiple communication passes achieve the 96% data continuity target.

**통신 운용 / Communication operations:**
- 3 short passes (1.6 hrs each) + 1 long pass (8 hrs) per day
- 연속 전송률 / Continuous rate: 40 kbit/s
- MDI 고속 모드 / MDI high rate: 160 kbit/s (during long pass, ~60 days/year)
- 온보드 저장 / On-board storage: 2 Gbit SSR + 1 Gbit tape recorder

In [ ]:
# --- Telemetry Budget Calculation ---

# Communication parameters.
continuous_rate = 40       # kbit/s (normal science telemetry)
mdi_high_rate = 160        # kbit/s (MDI high-rate mode)
short_pass_hrs = 1.6       # hours per short pass
n_short_passes = 3         # per day
long_pass_hrs = 8.0        # hours per long pass
n_long_passes = 1          # per day
mdi_high_days_per_year = 60  # days/year with MDI high-rate

# On-board storage.
ssr_capacity_gbit = 2.0     # Solid State Recorder
tape_capacity_gbit = 1.0    # Tape recorder
total_storage_gbit = ssr_capacity_gbit + tape_capacity_gbit

# --- Daily data volume ---
# Total contact time per day.
total_contact_hrs = n_short_passes * short_pass_hrs + n_long_passes * long_pass_hrs
total_contact_sec = total_contact_hrs * 3600

# Data downlinked per day during normal operations.
daily_downlink_kbit = continuous_rate * total_contact_sec
daily_downlink_gbit = daily_downlink_kbit / 1e6

# Data generated continuously (24 hours).
daily_generated_kbit = continuous_rate * 24 * 3600
daily_generated_gbit = daily_generated_kbit / 1e6

# Data continuity: fraction of time in contact.
contact_fraction = total_contact_hrs / 24

# MDI high-rate additional data (during long pass only).
mdi_extra_rate = mdi_high_rate - continuous_rate  # Additional rate
mdi_daily_extra_kbit = mdi_extra_rate * long_pass_hrs * 3600
mdi_daily_extra_gbit = mdi_daily_extra_kbit / 1e6

# Yearly totals.
normal_days = 365 - mdi_high_days_per_year
yearly_normal_gbit = normal_days * daily_downlink_gbit
yearly_mdi_gbit = mdi_high_days_per_year * (daily_downlink_gbit + mdi_daily_extra_gbit)
yearly_total_gbit = yearly_normal_gbit + yearly_mdi_gbit

print("=== SOHO Telemetry Budget ===\n")
print(f"Daily contact time: {total_contact_hrs:.1f} hrs "
      f"({n_short_passes}x{short_pass_hrs}h + {n_long_passes}x{long_pass_hrs}h)")
print(f"Contact fraction: {contact_fraction*100:.1f}%")
print(f"\nDaily data volume (normal): {daily_downlink_gbit:.2f} Gbit")
print(f"Daily data generated (24h): {daily_generated_gbit:.2f} Gbit")
print(f"On-board storage: {total_storage_gbit:.1f} Gbit "
      f"(covers {total_storage_gbit/daily_generated_gbit*24:.1f} hrs of data)")
print(f"\nMDI high-rate extra per day: {mdi_daily_extra_gbit:.2f} Gbit")
print(f"\n--- Yearly Totals ---")
print(f"Normal days ({normal_days}): {yearly_normal_gbit:.1f} Gbit")
print(f"MDI high-rate days ({mdi_high_days_per_year}): {yearly_mdi_gbit:.1f} Gbit")
print(f"Total yearly: {yearly_total_gbit:.1f} Gbit = {yearly_total_gbit/8:.1f} GByte")

# --- Compare with modern missions ---
sdo_daily_tbit = 1.5  # SDO: ~1.5 Tbit/day
soho_daily_gbit_avg = yearly_total_gbit / 365

print(f"\n--- Modern Comparison ---")
print(f"SOHO average: {soho_daily_gbit_avg:.1f} Gbit/day")
print(f"SDO:          {sdo_daily_tbit * 1000:.0f} Gbit/day")
print(f"SDO/SOHO ratio: {sdo_daily_tbit * 1000 / soho_daily_gbit_avg:.0f}x")

In [ ]:
# --- Visualize daily telemetry operations ---
# Show a 24-hour timeline of DSN contact and data flow.

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# --- Top: 24-hour DSN contact timeline ---
ax_time = axes[0]
hours = np.linspace(0, 24, 1000)

# Define contact windows (approximate schedule).
# 3 short passes spread through the day + 1 long pass.
contacts = [
    {'start': 0, 'end': 1.6, 'type': 'short', 'label': 'Short 1'},
    {'start': 6, 'end': 7.6, 'type': 'short', 'label': 'Short 2'},
    {'start': 12, 'end': 13.6, 'type': 'short', 'label': 'Short 3'},
    {'start': 16, 'end': 24, 'type': 'long', 'label': 'Long Pass (8h)'},
]

for c in contacts:
    color = '#42A5F5' if c['type'] == 'short' else '#66BB6A'
    ax_time.axvspan(c['start'], c['end'], alpha=0.4, color=color)
    ax_time.text((c['start'] + c['end']) / 2, 0.5, c['label'],
                 ha='center', va='center', fontsize=9, fontweight='bold')

# Data rate during contact.
rate_profile = np.zeros_like(hours)
for c in contacts:
    mask = (hours >= c['start']) & (hours <= c['end'])
    rate_profile[mask] = continuous_rate

ax_time.plot(hours, rate_profile, 'b-', linewidth=2, label='Downlink rate')
ax_time.set_xlabel('Hour of Day [UTC]')
ax_time.set_ylabel('Downlink Rate [kbit/s]')
ax_time.set_title('SOHO Daily DSN Contact Schedule')
ax_time.set_xlim(0, 24)
ax_time.set_ylim(-5, 200)
ax_time.axhline(y=continuous_rate, color='gray', linestyle='--', alpha=0.5,
                label=f'Continuous: {continuous_rate} kbit/s')
ax_time.legend(loc='upper right')
ax_time.grid(True, alpha=0.3)

# --- Bottom: On-board storage fill level ---
ax_storage = axes[1]

# Simulate storage: data accumulates when not in contact, drains during contact.
dt_hr = hours[1] - hours[0]
storage = np.zeros_like(hours)
gen_rate_kbit_per_hr = continuous_rate * 3600  # kbit generated per hour
down_rate_kbit_per_hr = continuous_rate * 3600  # kbit downlinked per hour

for i in range(1, len(hours)):
    h = hours[i]
    # Always generating data.
    storage[i] = storage[i-1] + gen_rate_kbit_per_hr * dt_hr

    # During contact, also downlinking.
    in_contact = any(c['start'] <= h <= c['end'] for c in contacts)
    if in_contact:
        # Downlink: can send real-time + stored data.
        # Real-time data goes directly, stored data also drains.
        storage[i] = max(0, storage[i] - 2 * down_rate_kbit_per_hr * dt_hr)

# Convert to Gbit.
storage_gbit = storage / 1e6

ax_storage.plot(hours, storage_gbit, 'r-', linewidth=2, label='SSR Fill Level')
ax_storage.axhline(y=total_storage_gbit, color='red', linestyle='--', alpha=0.5,
                   label=f'SSR Capacity: {total_storage_gbit:.0f} Gbit')
ax_storage.set_xlabel('Hour of Day [UTC]')
ax_storage.set_ylabel('Stored Data [Gbit]')
ax_storage.set_title('On-board Storage Usage Over 24 Hours')
ax_storage.set_xlim(0, 24)
ax_storage.legend(loc='upper left')
ax_storage.grid(True, alpha=0.3)

# Shade contact windows on storage plot too.
for c in contacts:
    color = '#42A5F5' if c['type'] == 'short' else '#66BB6A'
    ax_storage.axvspan(c['start'], c['end'], alpha=0.15, color=color)

plt.tight_layout()
plt.show()

# 96% continuity explanation.
print("96% data continuity target:")
print(f"  Contact fraction: {contact_fraction*100:.1f}% of 24 hours")
print(f"  On-board storage bridges the gaps: {total_storage_gbit} Gbit")
print(f"  This covers ~{total_storage_gbit/daily_generated_gbit*24:.0f} hours of buffered data")
print(f"  Combined contact + storage = effective ~96% continuity")

## Part 6: Solar Cycle Coverage / 태양 주기 커버리지

SOHO는 원래 2년 미션으로 계획되었지만, 30년 이상 운용되며 태양 주기 23, 24, 25를 커버했습니다 (Fig. 11 개념 재현).

SOHO was originally planned as a 2-year mission but operated for 30+ years, covering solar cycles 23, 24, and 25 (reproducing the concept of Fig. 11).

**주요 사건 / Key events:**
- 1995년 12월: 발사 / Launch (Dec 1995)
- 1996년 2월: L1 도착 / L1 arrival (Feb 1996)
- 1998년 6월: 자세 상실 / Attitude loss (Jun 1998)
- 1998년 9월: 복구 / Recovery (Sep 1998)
- 2003년: 마지막 자이로 고장 / Last gyro failure
- 2010년: SDO 발사 / SDO launch

In [ ]:
# --- Solar Cycle Model and SOHO Timeline ---
# Simple sinusoidal model for the ~11-year solar cycle.
# Sunspot number approximation using a sinusoidal with varying amplitude.

# Solar cycle parameters (approximate):
# Cycle 22: peak ~1989, max SSN ~160
# Cycle 23: peak ~2001, max SSN ~120
# Cycle 24: peak ~2014, max SSN ~80
# Cycle 25: peak ~2025, max SSN ~115 (ongoing)

years = np.linspace(1985, 2026, 2000)

def solar_cycle_model(t, t_min, t_peak, ssn_max):
    """Simple asymmetric solar cycle model.

    Args:
        t: Time array [years].
        t_min: Cycle minimum time [year].
        t_peak: Cycle maximum time [year].
        ssn_max: Peak sunspot number.

    Returns:
        Sunspot number array.
    """
    cycle_length = 11.0
    # Rise time is shorter than decay.
    rise_time = t_peak - t_min
    decay_time = cycle_length - rise_time
    t_next_min = t_min + cycle_length

    result = np.zeros_like(t)
    for i, ti in enumerate(t):
        if ti < t_min or ti > t_next_min:
            result[i] = 0
        elif ti <= t_peak:
            # Rising phase (faster).
            phase = (ti - t_min) / rise_time
            result[i] = ssn_max * np.sin(np.pi / 2 * phase)
        else:
            # Declining phase (slower).
            phase = (ti - t_peak) / decay_time
            result[i] = ssn_max * np.cos(np.pi / 2 * phase)
    return np.maximum(result, 0)


# Model each cycle.
ssn_22 = solar_cycle_model(years, 1986, 1989.5, 160)
ssn_23 = solar_cycle_model(years, 1996, 2001.5, 120)
ssn_24 = solar_cycle_model(years, 2008.5, 2014, 80)
ssn_25 = solar_cycle_model(years, 2019.5, 2025, 115)

ssn_total = ssn_22 + ssn_23 + ssn_24 + ssn_25

# --- Plot ---
fig, ax = plt.subplots(figsize=(16, 7))

# Solar cycle background.
ax.fill_between(years, ssn_total, alpha=0.3, color='orange', label='Sunspot Number (model)')
ax.plot(years, ssn_total, 'orange', linewidth=1.5)

# SOHO operation periods.
# Planned 2-year mission.
ax.axvspan(1995.95, 1998.0, alpha=0.2, color='blue', label='Planned 2-year mission')
# 6-year consumables lifetime.
ax.axvspan(1995.95, 2002.0, alpha=0.1, color='green', label='6-year consumables')
# Actual 30-year operation.
ax.axvspan(1995.95, 2025.95, alpha=0.08, color='red', label='Actual operation (30 years)')

# Key events.
events = [
    (1995.95, 'Launch\n(Dec 1995)', 'green'),
    (1996.15, 'L1 Arrival\n(Feb 1996)', 'blue'),
    (1998.45, 'Attitude Loss\n(Jun 1998)', 'red'),
    (1998.75, 'Recovery\n(Sep 1998)', 'darkgreen'),
    (2003.0, 'Last Gyro\nFailure', 'orange'),
    (2010.15, 'SDO Launch\n(Feb 2010)', 'purple'),
]

for year, label, color in events:
    ax.axvline(x=year, color=color, linestyle='--', linewidth=1.5, alpha=0.7)
    # Stagger vertical positions to avoid overlap.
    y_pos = 170 if events.index((year, label, color)) % 2 == 0 else 150
    ax.text(year, y_pos, label, ha='center', va='bottom', fontsize=8,
            fontweight='bold', color=color,
            bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.8))

# Solar cycle labels.
ax.text(2001, 130, 'Cycle 23', ha='center', fontsize=12, style='italic', color='gray')
ax.text(2014, 90, 'Cycle 24', ha='center', fontsize=12, style='italic', color='gray')
ax.text(2025, 125, 'Cycle 25', ha='center', fontsize=12, style='italic', color='gray')

ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('Sunspot Number', fontsize=12)
ax.set_title('SOHO Mission Timeline vs Solar Activity (cf. Fig. 11)', fontsize=14)
ax.set_xlim(1985, 2026)
ax.set_ylim(0, 200)
ax.legend(loc='upper left', fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("SOHO covered nearly 3 complete solar cycles:")
print("  Cycle 23 (1996-2008): Full coverage including maximum")
print("  Cycle 24 (2008-2019): Full coverage")
print("  Cycle 25 (2019-present): Rising phase and maximum")
print(f"\nTotal operation: {2025.95 - 1995.95:.0f} years (planned: 2 years)")

## Summary / 요약

SOHO와 이전/이후 태양 관측 미션을 비교합니다.

Comparison of SOHO with predecessor and successor solar observation missions.

In [ ]:
# --- Summary Comparison Table ---
# SOHO vs predecessors and successors.

summary_data = {
    'Mission': ['Skylab/ATM', 'SMM', 'SOHO', 'SDO', 'Solar Orbiter',
                'Parker Solar Probe'],
    'Launch': [1973, 1980, 1995, 2010, 2020, 2018],
    'Orbit': ['LEO', 'LEO', 'L1 Halo', 'GEO (inclined)', 'Heliocentric',
              'Heliocentric'],
    'Instruments': [8, 7, 12, 3, 10, 4],
    'Planned Life [yr]': ['<1', 2, 2, 5, 7, 7],
    'Actual Life [yr]': ['<1', 10, '30+', '15+', '6+', '7+'],
    'Data Rate': ['Film', '~5 kbit/s', '40 kbit/s', '~150 Mbit/s',
                  '~150 kbit/s', '~85 kbit/s'],
    'Key Innovation': [
        'First space coronagraph',
        'First solar max coverage',
        'L1 point, uninterrupted view',
        'High-cadence full-disk EUV',
        'Close-up, out of ecliptic',
        'In-situ at <10 Rs',
    ],
}

# Print as formatted table.
print("=" * 120)
print(f"{'Mission':<20} {'Launch':<8} {'Orbit':<16} {'Instr.':<8} "
      f"{'Planned':<10} {'Actual':<10} {'Data Rate':<16} {'Key Innovation'}")
print("=" * 120)
for i in range(len(summary_data['Mission'])):
    print(f"{summary_data['Mission'][i]:<20} "
          f"{summary_data['Launch'][i]:<8} "
          f"{summary_data['Orbit'][i]:<16} "
          f"{summary_data['Instruments'][i]:<8} "
          f"{str(summary_data['Planned Life [yr]'][i]):<10} "
          f"{str(summary_data['Actual Life [yr]'][i]):<10} "
          f"{summary_data['Data Rate'][i]:<16} "
          f"{summary_data['Key Innovation'][i]}")
print("=" * 120)

# --- Data rate evolution visualization ---
fig, ax = plt.subplots(figsize=(12, 5))

missions_rate = {
    'Skylab (1973)': 0.001,       # Film-based, ~equivalent
    'SMM (1980)': 5,
    'SOHO (1995)': 40,
    'STEREO (2006)': 720,
    'SDO (2010)': 150_000,
    'Solar Orbiter (2020)': 150,
    'PSP (2018)': 85,
}

x_pos = range(len(missions_rate))
bars = ax.bar(x_pos, list(missions_rate.values()),
              color=['#90A4AE', '#78909C', '#FF9800', '#42A5F5',
                     '#66BB6A', '#CE93D8', '#EF5350'],
              edgecolor='black', linewidth=0.5)
ax.set_xticks(list(x_pos))
ax.set_xticklabels(list(missions_rate.keys()), rotation=30, ha='right', fontsize=9)
ax.set_yscale('log')
ax.set_ylabel('Data Rate [kbit/s]')
ax.set_title('Solar Mission Data Rate Evolution')
ax.grid(True, axis='y', alpha=0.3)

# Add value labels.
for bar, val in zip(bars, missions_rate.values()):
    label = f'{val:.0f}' if val >= 1 else f'{val}'
    ax.text(bar.get_x() + bar.get_width() / 2, val * 1.5,
            f'{label} kbit/s', ha='center', va='bottom', fontsize=8,
            fontweight='bold')

# Highlight SOHO.
bars[2].set_edgecolor('red')
bars[2].set_linewidth(2)

plt.tight_layout()
plt.show()

print("\nSOHO's 40 kbit/s was a major step up from SMM's ~5 kbit/s,")
print("but SDO's ~150 Mbit/s represents a 3750x increase over SOHO.")
print("Deep-space missions (Solar Orbiter, PSP) are bandwidth-limited")
print("to ~100 kbit/s due to distance.")